# Multimodal EEG + EMG + Vision + Language Inference

This notebook integrates:
1. EEG gesture recognition
2. EMG gesture recognition
3. Vision-based hand-object understanding
4. Task instruction
5. LLM-based multimodal reasoning
6. Deterministic decision fusion


# Section 1 — Project setup

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import sys
import json
import numpy as np
import torch
import torch.nn as nn

from PIL import Image
from scipy.io import loadmat

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/NITT/Sem 5/PWPI/"
    "Multimodal_EEG_EMG_VLM_Project"
)

CODE_DIR = PROJECT_ROOT / "03_Code"
RESULTS_DIR = PROJECT_ROOT / "04_Results"

EEG_CODE_DIR = CODE_DIR / "EEG"
EMG_CODE_DIR = CODE_DIR / "EMG"
VISION_CODE_DIR = CODE_DIR / "Vision"
PROMPT_CODE_DIR = CODE_DIR / "Prompt"

for path in [CODE_DIR, EEG_CODE_DIR, EMG_CODE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project
Device: cuda


In [3]:

from EEG.eeg_decoder import EEGDecoder
from EEG.eeg_inference import infer_eeg

from Prompt.mappings import (
    EEG_GESTURE_MAP,
    EMG_LABEL_MAP
)


EEG_MODEL_PATH = PROJECT_ROOT / "05_Models/EEG/eeg_decoder_best_model.pth"

eeg_model = EEGDecoder(num_classes=7).to(DEVICE)

checkpoint = torch.load(
    EEG_MODEL_PATH,
    map_location=DEVICE
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    eeg_model.load_state_dict(checkpoint["model_state_dict"])
else:
    eeg_model.load_state_dict(checkpoint)

eeg_model.eval()

print("Existing EEGDecoder loaded successfully")
print("Checkpoint:", EEG_MODEL_PATH)

Existing EEGDecoder loaded successfully
Checkpoint: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/05_Models/EEG/eeg_decoder_best_model.pth


# Section 3 — EEG preprocessing and inference
### Raw EEG → shape normalization → model → gesture + confidence

In [4]:
EEG_RAW_DIR = PROJECT_ROOT / "01_Data/EEG/raw"

eeg_files = list(EEG_RAW_DIR.rglob("*.mat")) + \
            list(EEG_RAW_DIR.rglob("*.csv"))

print("EEG files found:", len(eeg_files))

if eeg_files:
    print("Example:", eeg_files[0])

EEG files found: 2772
Example: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/EEG/raw/mat_data/subject_1/S1_R1_G1.mat


## TEST

In [5]:
eeg_test_result = infer_eeg(eeg_files[0],model=eeg_model,device=DEVICE)
print(json.dumps(eeg_test_result, indent=2))

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


{
  "gesture": "medium_diameter_grasp",
  "class_id": 1,
  "confidence": 0.7448,
  "file": "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/EEG/raw/mat_data/subject_1/S1_R1_G1.mat"
}


# EMG PART


In [6]:


from EMG.emg_model import EMGNet
from EMG.emg_inference import infer_emg

# Load saved EMG model

In [7]:
EMG_MODEL_PATH = PROJECT_ROOT / "05_Models/EMG/emgnet_model.pth"

emg_model = EMGNet(out_gesture=4).to(DEVICE)

emg_checkpoint = torch.load(
    EMG_MODEL_PATH,
    map_location=DEVICE
)

emg_model.load_state_dict(emg_checkpoint)
emg_model.eval()

print("EMG model loaded successfully")
print("Checkpoint:", EMG_MODEL_PATH)

EMG model loaded successfully
Checkpoint: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/05_Models/EMG/emgnet_model.pth


# TEST EMG

In [8]:
EMG_RAW_DIR = PROJECT_ROOT / "01_Data/EMG/raw"

emg_files = list(EMG_RAW_DIR.rglob("*.mat"))

print("EMG files found:", len(emg_files))

if emg_files:
    print("Example:", emg_files[0])

emg_test_result = infer_emg(
    file_path=emg_files[0],
    model=emg_model,
    device=DEVICE
)

print(json.dumps(emg_test_result, indent=2))

EMG files found: 1386
Example: /content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/EMG/raw/subject_22/S22_R1_G1.mat
{
  "gesture": "large_diameter_grasp",
  "class_id": 0,
  "confidence": 0.9867,
  "num_windows": 88,
  "file": "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/EMG/raw/subject_22/S22_R1_G1.mat"
}


In [9]:
import sys
import inspect

sys.path.insert(0, str(PROJECT_ROOT / "03_Code"))

from Prompt.context import build_multimodal_context
from Prompt.prompt_builder import *
from Prompt.llm_reasoner import *
from Prompt.fusion import *

print("build_multimodal_context:")
print(inspect.signature(build_multimodal_context))

print("\nLLM reasoner functions:")
print([x for x in dir() if not x.startswith("_")])

build_multimodal_context:
(task_instruction, eeg_output, emg_output, vision_output)

LLM reasoner functions:
['CODE_DIR', 'DEFAULT_MODEL', 'DEVICE', 'EEGDecoder', 'EEG_CODE_DIR', 'EEG_GESTURE_MAP', 'EEG_MODEL_PATH', 'EEG_RAW_DIR', 'EMGNet', 'EMG_CODE_DIR', 'EMG_LABEL_MAP', 'EMG_MODEL_PATH', 'EMG_RAW_DIR', 'Image', 'In', 'OUTPUT_INSTRUCTIONS', 'Out', 'PROJECT_ROOT', 'PROMPT_CODE_DIR', 'PROMPT_VERSION', 'Path', 'REASONING_SCHEMA', 'RESULTS_DIR', 'SYSTEM_ROLE', 'VISION_CODE_DIR', 'build_multimodal_context', 'build_reasoning_prompt', 'checkpoint', 'create_gemini_client', 'deterministic_fusion', 'drive', 'eeg_files', 'eeg_model', 'eeg_test_result', 'emg_checkpoint', 'emg_files', 'emg_model', 'emg_test_result', 'exit', 'genai', 'get_ipython', 'infer_eeg', 'infer_emg', 'inspect', 'json', 'loadmat', 'neural_gesture_agreement', 'nn', 'normalize_text', 'np', 'object_matches_instruction', 'path', 'quit', 'run_reasoning', 'sys', 'torch', 'vision_has_active_interaction', 'vision_has_object']


In [10]:
import inspect

functions_to_check = [
    build_reasoning_prompt,
    run_reasoning,
    deterministic_fusion,
    neural_gesture_agreement,
    vision_has_object,
    vision_has_active_interaction,
    object_matches_instruction,
]

for fn in functions_to_check:
    print(f"\n{fn.__name__}{inspect.signature(fn)}")


build_reasoning_prompt(multimodal_context)

run_reasoning(client, prompt, model='gemini-3.1-flash-lite')

deterministic_fusion(multimodal_context)

neural_gesture_agreement(eeg_output, emg_output)

vision_has_object(vision_output)

vision_has_active_interaction(vision_output)

object_matches_instruction(instruction, vision_output)


In [11]:
import json

VISION_RESULTS_PATH = (
    PROJECT_ROOT / "04_Results/Vision/hands23_vision_predictions.json"
)

with open(VISION_RESULTS_PATH, "r") as f:
    vision_predictions = json.load(f)

print("Type:", type(vision_predictions))
print("Number of predictions:", len(vision_predictions))

# Inspect one prediction
print(json.dumps(vision_predictions[0], indent=2))

Type: <class 'list'>
Number of predictions: 26
{
  "image_name": "AR__4R6QekTdrw_9_2070_15.jpg",
  "image_path": "/content/drive/MyDrive/NITT/Sem 5/PWPI/Multimodal_EEG_EMG_VLM_Project/01_Data/Image/hands23/jpg/AR__4R6QekTdrw_9_2070_15.jpg",
  "prediction": {
    "hands": [
      {
        "handedness": "right",
        "contact_state": "no_contact",
        "primary_object": "unknown",
        "hand_object_relation": "unknown",
        "grasp_type": "unknown"
      }
    ]
  },
  "status": "success"
}


In [14]:
import inspect

from Vision.vision_inference import infer_vision
from Prompt.context import (
    build_vision_context,
    build_multimodal_context,
)

print("infer_vision:")
print(inspect.signature(infer_vision))

print("\nbuild_vision_context:")
print(inspect.signature(build_vision_context))

print("\nbuild_multimodal_context:")
print(inspect.signature(build_multimodal_context))

infer_vision:
(client, image)

build_vision_context:
(hands=None, handedness=None, contact_state=None, grasp_type=None, primary_object=None, secondary_object=None, hand_object_relation=None)

build_multimodal_context:
(task_instruction, eeg_output, emg_output, vision_output)
